# 1. Setup Paths and Source URLs

In [6]:
from pathlib import Path

import polars as pl

root = Path.cwd().resolve()
data_dir = root/ "data"
data_dir.mkdir(parents=True, exist_ok=True)

daioe_source: str = (
    "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_years/development/"
    "data/daioe_scb_years_all_levels.parquet"
)

scb_source: str = (
        "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_months_v2/daioe_pull/"
        "data/scb_months.parquet"

)

# 2. Load DAIOE and SCB Lazily

In [7]:
daioe_lf = pl.scan_parquet(
    daioe_source,
)

scb_lf = pl.scan_parquet(
    scb_source,
)

# 3. Quick Sanity Checks

Data preview

In [8]:
print(daioe_lf.head(5).collect())

shape: (5, 65)
┌───────┬───────────┬───────┬───────┬───┬──────────────┬──────────────┬──────────────┬─────────────┐
│ level ┆ ssyk_code ┆ age   ┆ sex   ┆ … ┆ daioe_lngmod ┆ daioe_transl ┆ daioe_speech ┆ daioe_genai │
│ ---   ┆ ---       ┆ ---   ┆ ---   ┆   ┆ _Level_Expos ┆ at_Level_Exp ┆ rec_Level_Ex ┆ _Level_Expo │
│ str   ┆ str       ┆ str   ┆ str   ┆   ┆ ure          ┆ osure        ┆ posure       ┆ sure        │
│       ┆           ┆       ┆       ┆   ┆ ---          ┆ ---          ┆ ---          ┆ ---         │
│       ┆           ┆       ┆       ┆   ┆ i8           ┆ i8           ┆ i8           ┆ i8          │
╞═══════╪═══════════╪═══════╪═══════╪═══╪══════════════╪══════════════╪══════════════╪═════════════╡
│ SSYK3 ┆ 732       ┆ 30-34 ┆ men   ┆ … ┆ 2            ┆ 3            ┆ 2            ┆ 2           │
│ SSYK3 ┆ 129       ┆ 50-54 ┆ men   ┆ … ┆ 4            ┆ 4            ┆ 4            ┆ 4           │
│ SSYK3 ┆ 152       ┆ 40-44 ┆ men   ┆ … ┆ 4            ┆ 4            ┆ 4   

In [9]:
print(scb_lf.head(5).collect())

shape: (5, 5)
┌────────┬─────┬──────────┬───────┬────────────┐
│ code_1 ┆ sex ┆ month    ┆ value ┆ occupation │
│ ---    ┆ --- ┆ ---      ┆ ---   ┆ ---        │
│ str    ┆ str ┆ str      ┆ str   ┆ str        │
╞════════╪═════╪══════════╪═══════╪════════════╡
│ 1      ┆ men ┆ 2015-Jan ┆ 135.1 ┆ Managers   │
│ 1      ┆ men ┆ 2015-Feb ┆ 125.6 ┆ Managers   │
│ 1      ┆ men ┆ 2015-Mar ┆ 120.2 ┆ Managers   │
│ 1      ┆ men ┆ 2015-Apr ┆ 141.6 ┆ Managers   │
│ 1      ┆ men ┆ 2015-May ┆ 137.1 ┆ Managers   │
└────────┴─────┴──────────┴───────┴────────────┘


In [ ]:
scb_lf_clean = scb_lf\
    .filter(pl.col("code_1").str.starts_with(0).not_())\
        .with_columns(
    pl.col("month")
      .str.extract(r"^(\d{4})", 1)
      .cast(pl.Int64)
      .alias("year"),
)


print(scb_lf_clean.limit(10).collect())

shape: (10, 6)
┌────────┬─────┬──────────┬───────┬────────────┬──────┐
│ code_1 ┆ sex ┆ month    ┆ value ┆ occupation ┆ year │
│ ---    ┆ --- ┆ ---      ┆ ---   ┆ ---        ┆ ---  │
│ str    ┆ str ┆ str      ┆ str   ┆ str        ┆ i64  │
╞════════╪═════╪══════════╪═══════╪════════════╪══════╡
│ 1      ┆ men ┆ 2015-Jan ┆ 135.1 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Feb ┆ 125.6 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Mar ┆ 120.2 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Apr ┆ 141.6 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-May ┆ 137.1 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Jun ┆ 112.8 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Jul ┆ 142.5 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Aug ┆ 137.1 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Sep ┆ 120.7 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Oct ┆ 150.0 ┆ Managers   ┆ 2015 │
└────────┴─────┴──────────┴───────┴────────────┴──────┘


# 4. Compute 1-month, 3-months and 6-months Change

In [24]:

change_keys = [col for col in scb_lf_clean.collect_schema().names() if col != "value"]
time_col = "month"
group_keys = [col for col in change_keys if col != time_col]

def _pct_chg(current: pl.Expr, shifted: pl.Expr) -> pl.Expr:
    return (
        pl.when(shifted.is_not_null() & shifted.ne(0))
        .then((current / shifted - 1) * 100)
        .otherwise(None)
    )


raw_emp = pl.col("value")
emp = pl.col("emp_count")

scb_lazy_lf_changes = (
    scb_lf_clean
    .with_columns(pl.col("value").replace("..", None).cast(pl.Float64))
    .group_by(change_keys)
    .agg(raw_emp.sum().alias("emp_count"))
    .with_columns(
        pl.col(time_col).str.strptime(pl.Date, "%Y-%b", strict=False).alias("_month_date"),
    )
    # Pre-compute shifted values
    .with_columns(
        emp.shift(1).over(group_keys, order_by="_month_date").alias("_emp_1m"),
        emp.shift(3).over(group_keys, order_by="_month_date").alias("_emp_3m"),
        emp.shift(6).over(group_keys, order_by="_month_date").alias("_emp_6m"),
    )
    # Absolute and percentage changes
    .with_columns(
        (emp - pl.col("_emp_1m")).alias("chg_1m"),
        (emp - pl.col("_emp_3m")).alias("chg_3m"),
        (emp - pl.col("_emp_6m")).alias("chg_6m"),
        _pct_chg(emp, pl.col("_emp_1m")).alias("pct_chg_1m"),
        _pct_chg(emp, pl.col("_emp_3m")).alias("pct_chg_3m"),
        _pct_chg(emp, pl.col("_emp_6m")).alias("pct_chg_6m"),
    )
    .drop("_month_date", "_emp_1m", "_emp_3m", "_emp_6m")
    .sort(
    by=[
        "code_1",
        "sex",
        "occupation",
        pl.col("month").str.strptime(pl.Date, "%Y-%b", strict=False),
    ],
)
)

scb_lazy_lf_changes.collect_schema()

Schema([('code_1', String),
        ('sex', String),
        ('month', String),
        ('occupation', String),
        ('year', Int64),
        ('emp_count', Float64),
        ('chg_1m', Float64),
        ('chg_3m', Float64),
        ('chg_6m', Float64),
        ('pct_chg_1m', Float64),
        ('pct_chg_3m', Float64),
        ('pct_chg_6m', Float64)])

In [25]:
scb_lazy_lf_changes.head(10).collect()

code_1,sex,month,occupation,year,emp_count,chg_1m,chg_3m,chg_6m,pct_chg_1m,pct_chg_3m,pct_chg_6m
str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64
"""1""","""men""","""2015-Jan""","""Managers""",2015,438.0,null,null,null,null,null,null
"""1""","""men""","""2015-Feb""","""Managers""",2015,412.7,-25.3,null,null,-5.776256,null,null
"""1""","""men""","""2015-Mar""","""Managers""",2015,393.7,-19.0,null,null,-4.603828,null,null
"""1""","""men""","""2015-Apr""","""Managers""",2015,452.1,58.4,14.1,null,14.83363,3.219178,null
"""1""","""men""","""2015-May""","""Managers""",2015,447.5,-4.6,34.8,null,-1.017474,8.432275,null
"""1""","""men""","""2015-Jun""","""Managers""",2015,372.2,-75.3,-21.5,null,-16.826816,-5.461011,null
"""1""","""men""","""2015-Jul""","""Managers""",2015,455.4,83.2,3.3,17.4,22.353573,0.729927,3.972603
"""1""","""men""","""2015-Aug""","""Managers""",2015,443.8,-11.6,-3.7,31.1,-2.547211,-0.826816,7.53574
"""1""","""men""","""2015-Sep""","""Managers""",2015,395.1,-48.7,22.9,1.4,-10.973411,6.152606,0.355601


# 5. Prepare DAIOE Level-1 Aggregates

In [19]:
weighted_daioe = daioe_lf\
    .filter(
        pl.col("level") == "SSYK1",
        )\
        .select(
            pl.col(["level", "ssyk_code", "year", "weight_sum"]),
            pl.col("^daioe_.*$"),
            pl.col("^pctl_daioe_.*$"),
            )\
            .group_by(["level", "ssyk_code", "year"])\
                .agg([
                    pl.col("weight_sum").mean().cast(pl.Int64),
                    pl.col("^daioe_.*$").mean(),
                    pl.col("^pctl_daioe_.*$").mean(),
                    ])

weighted_daioe.limit(10).collect()

level,ssyk_code,year,weight_sum,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SSYK1""","""5""",2022,1006464,25.651088,0.227595,4.485773,0.357588,0.13917,0.543948,0.180078,0.583915,0.031916,0.383761,1.918633,25.192383,0.225764,4.385275,0.351784,0.136949,0.541534,0.177666,0.575533,0.030758,0.368711,1.902243,2.0,1.0,2.0,1.0,2.0,2.0,3.0,3.0,3.0,3.0,3.0,37.5,12.5,25.0,12.5,50.0,37.5,50.0,50.0,50.0,50.0,50.0,25.0,12.5,25.0,12.5,37.5,37.5,50.0,50.0,50.0,50.0,50.0
"""SSYK1""","""8""",2024,347858,27.739882,0.247078,5.853991,0.390965,0.181807,0.519702,0.205744,0.502361,0.020999,0.45489,1.683537,27.923731,0.242952,5.752996,0.406446,0.189149,0.538344,0.209542,0.51159,0.021722,0.46303,1.728679,2.0,2.0,5.0,3.0,3.0,2.0,1.0,1.0,2.0,2.0,1.0,25.0,37.5,100.0,50.0,25.0,25.0,0.0,0.0,0.0,12.5,0.0,37.5,37.5,100.0,50.0,50.0,25.0,12.5,12.5,25.0,25.0,12.5
"""SSYK1""","""9""",2016,235399,12.368514,0.21964,3.56791,0.154175,0.035831,0.225512,0.072541,0.020552,0.005171,0.167438,0.347305,12.211664,0.21614,3.689088,0.151661,0.034402,0.209998,0.065042,0.018443,0.004502,0.149821,0.32139,1.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,75.0,0.0,0.0,0.0,25.0,37.5,37.5,37.5,0.0,0.0,0.0,75.0,12.5,0.0,0.0,0.0,0.0,0.0,12.5,0.0
"""SSYK1""","""2""",2015,973635,9.486693,0.240185,2.786063,0.200962,null,0.029438,0.006167,0.025969,0.003592,0.247376,0.092267,9.522744,0.242895,2.752531,0.201143,0.0,0.029889,0.006273,0.026659,0.003685,0.255934,0.094236,4.0,5.0,1.0,5.0,3.0,5.0,5.0,5.0,5.0,5.0,5.0,75.0,100.0,12.5,87.5,null,100.0,100.0,87.5,87.5,87.5,100.0,75.0,87.5,12.5,87.5,50.0,100.0,100.0,87.5,87.5,87.5,100.0
"""SSYK1""","""7""",2024,438278,27.935685,0.248149,5.735777,0.390714,0.185006,0.56162,0.2151,0.525917,0.021096,0.445916,1.791579,27.367265,0.244094,5.544717,0.386852,0.182734,0.563614,0.212293,0.515463,0.020876,0.438712,1.778487,2.0,3.0,5.0,2.0,2.0,3.0,2.0,2.0,1.0,1.0,2.0,37.5,50.0,87.5,37.5,37.5,50.0,12.5,12.5,12.5,0.0,37.5,25.0,50.0,87.5,37.5,25.0,50.0,25.0,25.0,12.5,0.0,37.5
"""SSYK1""","""2""",2020,1124063,26.997509,0.300468,4.379279,0.389043,0.094852,0.604288,0.238056,0.386659,0.043498,0.43064,1.718914,27.494918,0.307544,4.35286,0.393817,0.096568,0.621563,0.245696,0.402092,0.04506,0.448864,1.777621,5.0,5.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,100.0,12.5,87.5,87.5,100.0,100.0,87.5,87.5,87.5,100.0,87.5,87.5,12.5,87.5,87.5,100.0,100.0,100.0,87.5,87.5,100.0
"""SSYK1""","""1""",2015,276948,8.615596,0.217084,2.477278,0.18398,null,0.027156,0.005623,0.024023,0.003389,0.234964,0.085313,8.786716,0.223022,2.511899,0.189463,0.